In [ ]:
## Nikolay Vorontsov,
## Mushroom task
## Inference with fine-tuned model

In [ ]:
!pip install transformers huggingface_hub pip torch jsonlines regex
# Install necessary dependencies
!pip install transformers peft accelerate huggingface_hub
!pip install -q trl xformers wandb datasets einops sentencepiece
!pip install -U datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 52.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlink

In [ ]:
import jsonlines
import re
import torch
import json
from transformers import AutoTokenizer, AutoModelForTokenClassification

from huggingface_hub import login


from google.colab import userdata

HUGGING_API = userdata.get('HUGGINGFACE_READ_AND_WRITE')

In [ ]:

# Login to Hugging Face
login(token=HUGGING_API)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("nicksnlp/llama-7B-hallucination")
model = AutoModelForTokenClassification.from_pretrained("nicksnlp/llama-7B-hallucination")

# Check for CUDA availability and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

tokenizer_config.json:   0%|          | 0.00/978 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
`low_cpu_mem_usage` was None, now default to True since model is quantized.


model.safetensors:   0%|          | 0.00/3.91G [00:00<?, ?B/s]

In [ ]:

"""
def infer_with_model(input_text):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    predicted_labels = torch.argmax(logits, dim=-1)
    tokens = tokenizer.tokenize(input_text)
    labeled_tokens = list(zip(tokens, predicted_labels[0].tolist()))
    hallucinated_words = [token for token, label in labeled_tokens if label == 1]
    return hallucinated_words
"""
def infer_with_model(input_text):
    # Tokenize the input text
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=128)

    # Move input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Predict the token labels (hallucination vs. correct)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # Raw logits output from the model

    # Get the predicted labels (0 for correct, 1 for hallucinated)
    predicted_labels = torch.argmax(logits, dim=-1)

    # Decode the tokens from the input text
    tokens = tokenizer.tokenize(input_text)

    # Get the corresponding predicted labels for each token
    labeled_tokens = list(zip(tokens, predicted_labels[0].tolist()))

    # Create a list of hallucinated words
    hallucinated_words = [token for token, label in labeled_tokens if label == 1]

    return hallucinated_words, labeled_tokens


In [ ]:
# https://chatgpt.com/share/679a8019-8234-800b-a54c-b2d42bc02dea

# Convert token labels to character labels.
import regex as re  # Use 'regex' module for Unicode character classes

def export_spans(tokens, full_text, answer_text):

  # Step 1: Build the aligned text and character labels
  aligned_text = ""
  char_labels = ""
  special_tokens = set(tokenizer.all_special_tokens)

  def is_punctuation(token):
    return re.match(r"\p{P}+", token)  # Matches any Unicode punctuation

  for token, label in tokens:
      token_clean = tokenizer.decode(tokenizer.convert_tokens_to_ids(token)) #token.replace('▁', '')  # Remove subword marker
      ## THIS IS PROBABLY NOT A GOOD APPROACH
      if aligned_text and token.startswith('▁') and not ( token_clean.startswith(" ") or is_punctuation(token_clean) ):
          aligned_text += " "  # Preserve space between words
          char_labels += "0"  # Spaces should be labeled 0

      aligned_text += token_clean  # Append token text
      char_labels += str(label) * len(token_clean)  # Append corresponding labels

  # Step 2: Locate the `@@` marker in aligned_text
  trim_marker = "@@"
  marker_pos = aligned_text.find(trim_marker)

  if marker_pos == -1:
      raise ValueError("Trim marker '@TRIM@' not found in aligned text!")

  answer_start = marker_pos + len(trim_marker)

  # Step 3: Trim aligned_text and char_labels
  trimmed_aligned_text = aligned_text[answer_start:]
  trimmed_char_labels = char_labels[answer_start:]

  # Step 2: Convert character-level labels into spans
  spans = []
  start = None

  for i, char in enumerate(trimmed_char_labels):
      if char == '1':
          if start is None:
              start = i  # Start new span
      else:
          if start is not None:
              spans.append((start, i))  # End the current span
              start = None

  # If last label was '1', we need to close the span
  if start is not None:
      spans.append((start, len(trimmed_char_labels)))

  # Output results
  print(tokens)

  print("".join(t[0] for t in tokens))
  tokens_dec = []
  for t in tokens:
    t_dec = tokenizer.decode(tokenizer.convert_tokens_to_ids(t[0]))
    tokens_dec.append(t_dec)
  print("".join(t for t in tokens_dec))

  print("Aligned Text:", aligned_text)
  print("Spans of Label 1:", spans)

  print("Trimmed Aligned Text:", trimmed_aligned_text)
  print("Spans of Label 1:", spans)

  # Verify by extracting spans from the text
  for start, end in spans:
      print(f"answer_text[{start}:{end}] -> '{answer_text[start:end]}'")

  return trimmed_aligned_text, trimmed_char_labels, spans


In [ ]:
# https://chatgpt.com/share/679b56a8-c4ac-800b-a808-b771678cf9ad

import regex as re

def export_spans_using_offsets(tokens, full_text, tokenizer):
    tokenized = tokenizer(full_text, return_offsets_mapping=True, add_special_tokens=True)
    offset_mapping = tokenized["offset_mapping"]

    spans = []
    start = None

    for (token, label), (char_start, char_end) in zip(tokens, offset_mapping):
        if char_start is None or char_end is None:
            continue  # Skip special tokens

        if label == 1:
            if start is None:
                start = char_start  # Start a new span
        else:
            if start is not None:
                spans.append((start, char_start))  # Close span
                start = None

    if start is not None:
        spans.append((start, char_end))  # Close last span

    # Extract spans from text
    extracted_texts = [full_text[start:end] for start, end in spans]

    print("Aligned Text:", full_text)
    print("Spans of Label 1:", spans)
    print("Extracted Answer Texts:", extracted_texts)

    return full_text, spans, extracted_texts


In [ ]:
# Example usage of the inference function
question = "Which municipalities does the Italian commune of Ponzone border?"
input_text = " Ponza\n"
full_text = question+"@@"+input_text
hallucinated_words, labeled_tokens = infer_with_model(full_text)

# Print the list of hallucinated words
print("Hallucinated words:")
print(hallucinated_words)
print(list(tokenizer.decode(tokenizer.convert_tokens_to_ids(word)) for word in hallucinated_words))
print(full_text)
print(labeled_tokens)


Hallucinated words:
['▁does', '▁Pon', '▁border', '▁Pon']
['does', 'Pon', 'border', 'Pon']
Which municipalities does the Italian commune of Ponzone border?@@ Ponza

[('▁Which', 0), ('▁municipal', 0), ('ities', 0), ('▁does', 1), ('▁the', 0), ('▁Italian', 0), ('▁commune', 0), ('▁of', 0), ('▁Pon', 1), ('zone', 0), ('▁border', 1), ('?', 0), ('@@', 0), ('▁Pon', 1), ('za', 0), ('<0x0A>', 0)]


In [ ]:
t, ch, s = export_spans(labeled_tokens, full_text, input_text)
print(t)
print(ch)

[('▁Which', 0), ('▁municipal', 0), ('ities', 0), ('▁does', 1), ('▁the', 0), ('▁Italian', 0), ('▁commune', 0), ('▁of', 0), ('▁Pon', 1), ('zone', 0), ('▁border', 1), ('?', 0), ('@@', 0), ('▁Pon', 1), ('za', 0), ('<0x0A>', 0)]
▁Which▁municipalities▁does▁the▁Italian▁commune▁of▁Ponzone▁border?@@▁Ponza<0x0A>
WhichmunicipalitiesdoestheItaliancommuneofPonzoneborder?@@Ponza

Aligned Text: Which municipalities does the Italian commune of Ponzone border?@@ Ponza

Spans of Label 1: [(1, 4)]
Trimmed Aligned Text:  Ponza

Spans of Label 1: [(1, 4)]
answer_text[1:4] -> 'Pon'
 Ponza

0111000


In [ ]:
def test_inferences(validation_file, output_file):
  with jsonlines.open(validation_file) as reader, jsonlines.open(output_file, 'w') as writer:
    for datapoint in reader:
            model_output_text = datapoint.get("model_output_text", "")
            question = datapoint.get("model_input", "")
            qa_pair = question+"@@"+model_output_text

            hallucinated_words, labeled_tokens = infer_with_model(qa_pair)

            _, _, hard_labels = export_spans(labeled_tokens, qa_pair, model_output_text)

            datapoint["hard_labels"] = hard_labels
            #datapoint["id"] = datapoint["id"].replace('_unlabeled', '')

            writer.write(datapoint)

In [ ]:
# "a" creates the file if it doesn't exist
output_file = "/content/mushroom.en-tst.v1.jsonl_llama_7b-hallucinations"

with open(output_file, "a") as file:
    pass  # Do nothing, just ensure the file exists

print(f"{output_file} is created or already exists.")

/content/mushroom.en-tst.v1.jsonl_llama_7b-hallucinations is created or already exists.


In [ ]:

# Example usage:
validation_file = "/content/mushroom.en-tst.v1.jsonl" #"/content/mushroom.en-val.v2.unlabeled.jsonl"
test_inferences(validation_file, output_file)
print(f"Processed data written to {output_file}")

[('▁Did', 0), ('▁Alberto', 1), ('▁Fou', 1), ('illi', 1), ('oux', 0), ('▁ever', 1), ('▁play', 0), ('▁in', 1), ('▁a', 0), ('▁world', 0), ('▁cup', 1), ('▁championship', 0), ('?', 1), ('@@', 0), ('▁No', 1), (',', 1), ('▁Al', 0), ('bero', 1), ('▁Fou', 1), ('lo', 1), ('is', 1), ('▁was', 1), ('▁not', 1), ('▁in', 1), ('▁any', 1), ('▁of', 0), ('▁the', 0), ('▁FIFA', 1), ('▁World', 0), ('▁Cup', 0), ('▁final', 0), ('s', 1), ('.', 1), ('<0x0A>', 0)]
Aligned Text: Did Alberto Fou illi oux ever play in a world cup championship ? @@ No , Al bero Fou lo is was not in any of the FIFA World Cup final s . 

Spans of Label 1: [(1, 3), (4, 5), (9, 13), (14, 17), (18, 20), (21, 23), (24, 27), (28, 31), (32, 34), (35, 38), (46, 50), (67, 68), (69, 70)]
Trimmed Aligned Text:  No , Al bero Fou lo is was not in any of the FIFA World Cup final s . 

Spans of Label 1: [(1, 3), (4, 5), (9, 13), (14, 17), (18, 20), (21, 23), (24, 27), (28, 31), (32, 34), (35, 38), (46, 50), (67, 68), (69, 70)]
answer_text[1:3] -> 'N

ValueError: Trim marker '@TRIM@' not found in aligned text!

In [ ]:
# Clean_output

with open(output_file, "r", encoding='utf-8') as jsonl_file:
    lines = jsonl_file.readlines()

    for line in lines:

        # Remove '_unlabeled' from the 'id' field
        data_to_resave = json.loads(line)

        data_to_resave['id'] = data_to_resave['id'].replace('_unlabeled', '')
        print(data_to_resave['id'])
        print(data_to_resave["hard_labels"])

        soft_labels = [{'start': label[0], 'prob': float(1), 'end': label[1]} for label in data_to_resave['hard_labels'] if label]
        print(soft_labels)


             # Save the datapoint to the JSONL file
        with open(f"{output_file}_no_extra_keys_soft_labels_prob1.jsonl", "a", encoding='utf-8') as jsonl_file:
            datapoint_labelled = {
                "id":data_to_resave["id"],
                "lang":data_to_resave["lang"],
                "model_input":data_to_resave["model_input"],
                "model_output_text":data_to_resave["model_output_text"],
                "model_id":data_to_resave["model_id"],
                "soft_labels":soft_labels, #instead of data_to_resave["soft_labels"], that is to output an empty list.
                "hard_labels":data_to_resave["hard_labels"],
                "model_output_logits":data_to_resave["model_output_logits"],
                "model_output_tokens":data_to_resave["model_output_tokens"],
            }
            jsonl_file.write(json.dumps(datapoint_labelled) + "\n")

tst-en-1
[[1, 4], [7, 11], [12, 19], [20, 23], [24, 27], [28, 30], [31, 34], [42, 46], [62, 64]]
[{'start': 1, 'prob': 1.0, 'end': 4}, {'start': 7, 'prob': 1.0, 'end': 11}, {'start': 12, 'prob': 1.0, 'end': 19}, {'start': 20, 'prob': 1.0, 'end': 23}, {'start': 24, 'prob': 1.0, 'end': 27}, {'start': 28, 'prob': 1.0, 'end': 30}, {'start': 31, 'prob': 1.0, 'end': 34}, {'start': 42, 'prob': 1.0, 'end': 46}, {'start': 62, 'prob': 1.0, 'end': 64}]
tst-en-2
[[0, 5], [15, 18], [19, 23], [24, 26], [36, 37], [38, 45]]
[{'start': 0, 'prob': 1.0, 'end': 5}, {'start': 15, 'prob': 1.0, 'end': 18}, {'start': 19, 'prob': 1.0, 'end': 23}, {'start': 24, 'prob': 1.0, 'end': 26}, {'start': 36, 'prob': 1.0, 'end': 37}, {'start': 38, 'prob': 1.0, 'end': 45}]
tst-en-3
[[1, 7], [8, 9], [27, 34], [38, 41], [42, 49], [50, 52], [56, 58]]
[{'start': 1, 'prob': 1.0, 'end': 7}, {'start': 8, 'prob': 1.0, 'end': 9}, {'start': 27, 'prob': 1.0, 'end': 34}, {'start': 38, 'prob': 1.0, 'end': 41}, {'start': 42, 'prob': 1.